# Assignment — Bloc 1
### Mini-repte: Generar un to pur i modular-ne freqüència i amplitud

**Instruccions de muntatge:** Penja aquest fitxer a Google Drive, comparteix-lo com a còpia individual per alumne (Classroom ho fa automàticament si l'adjuntes com a "Assignment" amb l'opció "Crea una còpia per a cada alumne"). Pes: dins del 40% de workshops/mini-reptes.

**Objectiu:** Completa les funcions marcades amb `# TODO`. Cada part té una cel·la d'autotest — si no dona error i sents l'àudio correctament, ho has fet bé. Si el so "no sona com esperes", torna a revisar la funció.

No esborris les cel·les d'autotest.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio

sample_rate = 44100

## Part 1 — Generar un to pur

Completa la funció `generate_tone`. Ha de retornar un array NumPy amb una ona sinusoidal.

In [ ]:
def generate_tone(freq, duration, amplitude=0.5, sample_rate=44100):
    # TODO 1: crea l'eix de temps 't' amb np.linspace
    # ha d'anar de 0 a 'duration', amb sample_rate*duration mostres, endpoint=False
    t = None  # <-- substitueix aquesta línia

    # TODO 2: calcula 'wave' com una ona sinusoidal de freqüència 'freq'
    # multiplicada per 'amplitude'
    wave = None  # <-- substitueix aquesta línia

    return wave

In [ ]:
# AUTOTEST 1 — no modifiquis aquesta cel·la

test_wave = generate_tone(freq=440, duration=1.0, amplitude=0.5, sample_rate=44100)

assert test_wave is not None, "La funció retorna None: revisa els TODO"
assert isinstance(test_wave, np.ndarray), "El resultat ha de ser un np.array"
assert len(test_wave) == 44100, f"S'esperaven 44100 mostres, n'hi ha {len(test_wave)}"
assert np.max(np.abs(test_wave)) <= 0.5 + 1e-6, "L'amplitud màxima hauria de ser ~0.5"
assert np.max(np.abs(test_wave)) > 0.4, "L'amplitud sembla massa baixa, revisa el càlcul"

print("✅ Part 1 correcta!")
display(Audio(test_wave, rate=sample_rate))

## Part 2 — Modular la freqüència (sweep)

Ara farem que la freqüència canviï al llarg del temps (un "glissando"): comença a `freq_start` i acaba a `freq_end`.

**Pista:** en comptes d'un `freq` fix, necessites un array `freq` que canviï linealment al llarg de `t` (usa `np.linspace` també per a la freqüència). Després: `wave = amplitude * np.sin(2 * np.pi * freq * t)` — funciona igual, encara que `freq` sigui ara un array!

In [ ]:
def generate_sweep(freq_start, freq_end, duration, amplitude=0.5, sample_rate=44100):
    t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

    # TODO 3: crea un array 'freq' que vagi linealment de freq_start a freq_end
    # (mateixa longitud que 't')
    freq = None  # <-- substitueix aquesta línia

    # TODO 4: calcula 'wave' igual que abans, però amb aquest 'freq' variable
    wave = None  # <-- substitueix aquesta línia

    return wave

In [ ]:
# AUTOTEST 2 — no modifiquis aquesta cel·la

sweep_wave = generate_sweep(freq_start=220, freq_end=880, duration=2.0, amplitude=0.5)

assert sweep_wave is not None, "La funció retorna None: revisa els TODO"
assert len(sweep_wave) == 44100 * 2, f"S'esperaven 88200 mostres, n'hi ha {len(sweep_wave)}"

# Comprovem que la freqüència instantània puja: mirem la densitat de creuaments per zero
# a l'inici vs al final (heurística simple, no cal entendre el detall)
zero_crossings = np.where(np.diff(np.sign(sweep_wave)))[0]
first_half_density = np.sum(zero_crossings < len(sweep_wave) // 2)
second_half_density = np.sum(zero_crossings >= len(sweep_wave) // 2)
assert second_half_density > first_half_density, \
    "La freqüència no sembla augmentar al llarg del temps. Revisa l'array 'freq'."

print("✅ Part 2 correcta! Escolta com puja el to:")

plt.plot(sweep_wave[:2000])
plt.title("Sweep (primers 2000 mostres)")
plt.show()

display(Audio(sweep_wave, rate=sample_rate))

## Part 3 — Modular l'amplitud (fade in / fade out)

Ara fes que el so comenci en silenci, pugi de volum fins al màxim a la meitat, i torni a baixar fins al final (una "envolvent" triangular molt simple).

**Pista:** crea un array `envelope` que vagi de 0 a `amplitude` i torni a 0, amb la mateixa longitud que `t`. Pots fer-ho amb dos `np.linspace` concatenats amb `np.concatenate`. Després multiplica: `wave = envelope * np.sin(...)`

In [ ]:
def generate_tone_with_envelope(freq, duration, amplitude=0.5, sample_rate=44100):
    n_samples = int(sample_rate * duration)
    t = np.linspace(0, duration, n_samples, endpoint=False)

    # TODO 5: crea 'envelope': un array que va de 0 a 'amplitude' (primera meitat)
    # i de 'amplitude' a 0 (segona meitat). Mateixa longitud total que 't'.
    half = n_samples // 2
    rise = None   # <-- np.linspace(0, amplitude, half)
    fall = None   # <-- np.linspace(amplitude, 0, n_samples - half)
    envelope = None  # <-- np.concatenate([rise, fall])

    # TODO 6: aplica l'envolvent a l'ona sinusoidal
    wave = None  # <-- envelope * np.sin(2 * np.pi * freq * t)

    return wave

In [ ]:
# AUTOTEST 3 — no modifiquis aquesta cel·la

env_wave = generate_tone_with_envelope(freq=440, duration=2.0, amplitude=0.5)

assert env_wave is not None, "La funció retorna None: revisa els TODO"
assert len(env_wave) == 44100 * 2, f"Longitud incorrecta: {len(env_wave)}"

start_amp = np.max(np.abs(env_wave[:100]))
middle_amp = np.max(np.abs(env_wave[44090:44190]))
end_amp = np.max(np.abs(env_wave[-100:]))

assert start_amp < 0.05, f"L'inici hauria de ser gairebé silenci, amplitud={start_amp:.3f}"
assert end_amp < 0.05, f"El final hauria de ser gairebé silenci, amplitud={end_amp:.3f}"
assert middle_amp > 0.3, f"El centre hauria de tenir amplitud alta, té {middle_amp:.3f}"

print("✅ Part 3 correcta! Hauries de sentir un 'fade in / fade out':")

plt.plot(env_wave)
plt.title("Envolvent triangular")
plt.show()

display(Audio(env_wave, rate=sample_rate))

---
## 🚀 Challenges (opcional, nivell avançat)

Si has acabat i vols anar més enllà, prova alguna d'aquestes (sense autotest — escolta el resultat):

1. **Vibrato**: en comptes d'un sweep lineal, fes que la freqüència oscil·li ràpidament amunt i avall amb un altre `np.sin` (un LFO).
2. **Envolvent ADSR simple**: en comptes d'un triangle, crea una envolvent amb 4 trams (Attack, Decay, Sustain, Release) amb durades diferents.
3. **Combina-ho tot**: crea un so amb sweep de freqüència I envolvent d'amplitud alhora.
4. **Forma d'ona diferent**: substitueix `np.sin` per `scipy.signal.square` o `sawtooth` (vist a Patches) i compara el resultat.

In [ ]:
# El teu codi del Challenge aquí